# Data Challenge — Entrega 1: Definição e Análise Inicial

**Desafio 1 — Classificação de doenças em folhas de feijão**  
**Dataset:** Beans (AI-Lab Makerere / Hugging Face)

## 1. Problema e objetivo



## 2. Configuração do ambiente e reprodutibilidade

Importamos as bibliotecas utilizadas ao longo do notebook e as funções do pacote do projeto. A semente global (`SEED = 42`) é fixada para tornar as etapas reprodutíveis. As figuras e artefatos intermediários são organizados em diretórios próprios do repositório.


In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bean_leaf_disease.data.beans import (
    class_distribution,
    class_names,
    load_beans_dataset,
    make_tf_dataset,
    split_summary,
)
from bean_leaf_disease.utils.reproducibility import set_global_seed

SEED = 42
set_global_seed(SEED)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 3. Ingestão e validação inicial do dataset

O dataset `beans` é carregado a partir do Hugging Face por meio da função centralizada no pacote do projeto. Mantemos os splits originais de treino, validação e teste para que todas as etapas do grupo trabalhem sobre exatamente a mesma separação de dados.


In [ ]:
dataset = load_beans_dataset()
dataset

### 3.1 Resumo dos splits


In [ ]:
summary = split_summary(dataset)
summary

### 3.2 Classes disponíveis


In [ ]:
labels = class_names(dataset)
labels

## 4. Distribuição das classes por split
Um desbalanceamento forte tornaria a acurácia uma métrica enganosa e exigiria cuidados como pesos de classe ou reamostragem.

**Observações:**
- As três classes estão praticamente equilibradas em todos os splits: treino com 345, 348 e 341 imagens, validação com 44, 45 e 44, e teste com 43, 43 e 42. Cada classe representa cerca de 33% de cada split.
- A divisão é de aproximadamente 80% / 10% / 10% (1034, 133 e 128 imagens).
- Não há necessidade de pesos de classe nem de reamostragem, e a acurácia global não é enganosa neste caso. Ainda assim, o F1-macro e a matriz de confusão mostram em que classes o modelo erra.
- Validação e teste são pequenos (cerca de 130 imagens cada), então cada imagem pesa cerca de 0,75 ponto percentual na acurácia. Pequenas diferenças entre modelos podem ser ruído, e isso deve ser dito na análise de resultados.


In [ ]:
distribution = class_distribution(dataset)
display(distribution)

plt.figure(figsize=(9, 5))
sns.barplot(data=distribution, x="split", y="count", hue="class_name")
plt.title("Distribuição de classes por split")
plt.xlabel("Split"); plt.ylabel("Quantidade de imagens")
plt.tight_layout()
plt.savefig(FIG_DIR / "distribuicao_classes.png", dpi=150)
plt.show()

## 5. Inspeção visual das amostras
A grade mostra quatro exemplos de treino de cada classe e permite avaliar visualmente enquadramento, iluminação, fundo e características aparentes das doenças.

**Observações** (com base nos quatro primeiros exemplos de treino de cada classe):
- São fotografias de campo, não imagens de laboratório: o fundo é solo avermelhado, mato e outras folhas, e em vários casos há uma mão segurando a folha. A folha aparece em enquadramentos, escalas e orientações bem diferentes, e a iluminação varia (sol direto, sombra projetada, tons mais escuros).
- `healthy` tem folhas de verde uniforme, sem manchas.
- `angular_leaf_spot` apresenta lesões bem visíveis, marrom-avermelhadas ou escuras, espalhadas pela folha e, em vários exemplos, cercadas por um halo amarelado.
- `bean_rust` apresenta, nestes exemplos, sinais mais discretos: pontos pequenos amarelos ou marrons, esparsos, sobre folhas predominantemente verdes. Pode ser mais difícil de distinguir de `healthy` do que `angular_leaf_spot`.
- As lesões ocupam uma área pequena da imagem, e a redução para 224 × 224 pode apagar parte desse detalhe, principalmente em `bean_rust`.
- Folhas saudáveis às vezes aparecem ao lado de folhas amareladas ou manchadas ao fundo, o que pode confundir o modelo.
- Mãos, unhas pintadas e o tom do solo variam entre as fotos e não estão ligados à doença. O modelo pode aprender esses elementos como atalho, e a augmentation (rotação, flip, brilho e contraste) ajuda a reduzir essa dependência. Uma etapa posterior de explicabilidade com Grad-CAM poderá verificar se o modelo utiliza a folha e suas lesões, em vez de elementos do fundo, como evidência para a decisão.


In [ ]:
train_labels = np.array(dataset["train"]["labels"])

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, class_id in enumerate(range(len(labels))):
    idxs = np.where(train_labels == class_id)[0][:4]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(dataset["train"][int(idx)]["image"])
        axes[row, col].set_title(labels[class_id])
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "amostras_por_classe.png", dpi=150)
plt.show()

## 6. Propriedades das imagens
Calculamos, para todas as imagens dos três splits, a resolução original, a média dos canais R, G e B, o brilho (média em tons de cinza) e o contraste (desvio padrão em tons de cinza). Esses valores alimentam as análises seguintes e justificam o redimensionamento adotado neste notebook (224 × 224).

**Observações:**
- Todas as imagens têm a mesma resolução original, 500 × 500 pixels. Por serem quadradas, o redimensionamento para 224 × 224 não distorce a proporção da imagem.
- O brilho médio varia pouco entre as classes (de 0,42 a 0,46), mas o desvio padrão de cerca de 0,08 dentro de cada classe indica variação de iluminação entre as fotos.
- Os valores calculados aqui servem de referência para conferir a normalização usada no pipeline de pré-processamento.


In [ ]:
def img_stats(img):
    a = np.asarray(img.convert("RGB"), dtype=np.float32) / 255
    g = a.mean(axis=2)
    return dict(w=img.width, h=img.height,
                R=a[..., 0].mean(), G=a[..., 1].mean(), B=a[..., 2].mean(),
                brilho=g.mean(), contraste=g.std())

rows = []
for split in dataset:
    for ex in dataset[split]:
        d = img_stats(ex["image"])
        d["split"] = split
        d["classe"] = labels[ex["labels"]]
        rows.append(d)
stats = pd.DataFrame(rows)

print(stats[["w", "h"]].drop_duplicates())
display(stats.groupby("classe")[["R", "G", "B", "brilho", "contraste"]].agg(["mean", "std"]).round(3))

## 7. Brilho e contraste por classe
A variação de iluminação entre fotos de campo é um dos principais desafios do problema. Se brilho e contraste variam bastante dentro de cada classe, o modelo precisa ser robusto a isso, o que justifica augmentation de brilho e contraste.

**Observações:**
- **Brilho:** as medianas são próximas entre as classes (cerca de 0,43 a 0,46), com `healthy` ligeiramente mais clara. Dentro de cada classe, porém, a variação é grande: os valores vão de aproximadamente 0,2 a 0,7, ou seja, há fotos bem escuras e fotos bem claras em todas as classes.
- **Contraste:** as medianas ficam entre cerca de 0,15 e 0,18, com `angular_leaf_spot` mais baixa e `healthy` mais alta. Os intervalos se sobrepõem bastante, e a faixa total vai de cerca de 0,08 a 0,28.
- **Outliers:** são poucos. Aparecem alguns pontos isolados de contraste alto em `bean_rust` (cerca de 0,28) e um em `healthy`, e um ponto de brilho baixo em `angular_leaf_spot`.
- A variação dentro de cada classe é maior do que a diferença entre elas. Isso indica que brilho e contraste refletem principalmente as condições de captura (sol, sombra, câmera), e não a doença, e justifica augmentation de brilho e contraste.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=stats, x="classe", y="brilho", ax=ax[0]); ax[0].set_title("Brilho médio por classe")
sns.boxplot(data=stats, x="classe", y="contraste", ax=ax[1]); ax[1].set_title("Contraste por classe")
plt.tight_layout()
plt.savefig(FIG_DIR / "brilho_contraste.png", dpi=150)
plt.show()

## 8. Cor média por classe
As doenças alteram a coloração da folha, então a cor pode ser um sinal discriminativo. Ao mesmo tempo, um modelo que dependa demais da cor pode falhar com mudanças de iluminação ou de câmera, o que reforça a importância da augmentation.

**Observações:**
- `healthy` tem o maior verde (G ≈ 0,56) e o maior azul (B ≈ 0,34) entre as classes.
- `bean_rust` tem o menor verde (G ≈ 0,47), praticamente igual ao vermelho (R ≈ 0,48), o que combina com as manchas marrons e amareladas.
- `angular_leaf_spot` tem o maior vermelho (R ≈ 0,50) e o menor azul (B ≈ 0,29).
- A diferença de G entre `healthy` e `bean_rust` (cerca de 0,09) é da ordem de um desvio padrão (cerca de 0,085). Nos demais canais, as diferenças entre classes são menores que o desvio padrão. A cor média é, portanto, um sinal fraco, e as classes se sobrepõem bastante.
- A média é calculada sobre a imagem inteira, incluindo solo, mãos e outras folhas. Parte da diferença pode vir do fundo e não da folha, e isso só pode ser verificado com técnicas de explicabilidade, como o Grad-CAM em uma etapa posterior.


In [ ]:
cor = stats.groupby("classe")[["R", "G", "B"]].mean()
cor.plot(kind="bar", color=["#d62728", "#2ca02c", "#1f77b4"], figsize=(7, 4))
plt.title("Média dos canais RGB por classe")
plt.ylabel("Intensidade média [0-1]")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "cor_media_classe.png", dpi=150)
plt.show()

## 9. Duplicatas e vazamento entre splits
Imagens idênticas em treino e teste fazem o modelo ser avaliado em dados que ele já viu, inflando as métricas. Comparamos o hash MD5 de cada imagem dentro de cada split e entre os splits. O método detecta apenas cópias exatas, não imagens parecidas.

**Resultado:**
- Nenhuma duplicata interna em treino (1034 imagens), validação (133) ou teste (128).
- Nenhuma imagem em comum entre treino e validação, treino e teste, ou validação e teste.

**Conclusão:** os três splits são independentes quanto a cópias exatas, e as métricas de validação e teste não são inflacionadas por imagens repetidas. A checagem não cobre fotos diferentes da mesma planta ou da mesma folha, que podem existir em uma base coletada em campo. Isso é uma limitação conhecida, e o dataset não traz informação para verificá-la.


In [ ]:
def md5(img):
    return hashlib.md5(img.tobytes()).hexdigest()

hashes = {s: [md5(ex["image"]) for ex in dataset[s]] for s in dataset}

for s, h in hashes.items():
    print(f"{s}: {len(h)} imagens, {len(h) - len(set(h))} duplicatas internas")

print("treino ∩ validação:", len(set(hashes["train"]) & set(hashes["validation"])))
print("treino ∩ teste:", len(set(hashes["train"]) & set(hashes["test"])))
print("validação ∩ teste:", len(set(hashes["validation"]) & set(hashes["test"])))

## 10. Síntese da EDA e implicações para a metodologia

| Achado da EDA | Decisão proposta |
|---|---|
| Brilho e contraste variam muito dentro de cada classe (brilho de cerca de 0,2 a 0,7) | Augmentation de brilho e contraste |
| Fotos de campo, com folhas em enquadramentos, escalas e orientações diferentes | Rotação, flip horizontal e, se o grupo quiser testar, zoom ou recorte aleatório |
| Classes equilibradas (cerca de 33% cada) | Sem pesos de classe nem reamostragem; reportar acurácia global, F1-macro e matriz de confusão |
| Validação e teste pequenos (133 e 128 imagens) | Tratar diferenças pequenas entre modelos com cautela, pois cada imagem vale cerca de 0,75 ponto percentual |
| Cor média separa pouco as classes e o fundo (solo, mãos) varia | Usar augmentation para reduzir dependência do fundo e verificar com Grad-CAM em uma etapa posterior |
| Nenhum vazamento entre splits | Manter os splits originais do dataset |

Ponto a discutir: as imagens originais têm 500 × 500 pixels e as lesões ocupam uma área pequena. O redimensionamento para 224 × 224 reduz o detalhe das manchas, e vale testar uma resolução maior se o baseline tiver dificuldade com as classes doentes.


## 11. Metodologia de preparação dos dados

Com base na inspeção dos dados e nos requisitos de entrada dos modelos, adotamos inicialmente o redimensionamento das imagens para `224 × 224` pixels e a normalização dos valores de pixel para o intervalo `[0, 1]`.

A resolução de 224 × 224 oferece um ponto de partida computacionalmente viável para o baseline e é compatível com várias arquiteturas populares de Transfer Learning. Como a EDA mostrou que as imagens originais possuem 500 × 500 pixels e que algumas lesões ocupam regiões pequenas, resoluções maiores poderão ser avaliadas posteriormente caso o baseline apresente dificuldade em distinguir as classes.


In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = make_tf_dataset(
    dataset['train'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
val_ds = make_tf_dataset(
    dataset['validation'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
test_ds = make_tf_dataset(
    dataset['test'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

### 11.1 Validação do pipeline de imagens

A seguir inspecionamos um batch produzido pelo pipeline e verificamos forma dos tensores e faixa dos pixels após a normalização.


In [ ]:
images, y = next(iter(train_ds))
print('batch images:', images.shape)
print('batch labels:', y.shape)
print('min pixel:', float(images.numpy().min()))
print('max pixel:', float(images.numpy().max()))

### 11.2 Checks de qualidade

Os checks abaixo detectam problemas estruturais antes do treinamento, como ausência de splits, número inesperado de classes ou inconsistências no formato e normalização das imagens.


In [ ]:
assert set(dataset.keys()) >= {'train', 'validation', 'test'}
assert len(labels) == 3
assert images.shape[1:] == (224, 224, 3)
assert 0.0 <= float(images.numpy().min())
assert float(images.numpy().max()) <= 1.0
print('Checks iniciais concluídos com sucesso.')

## 12. Artefatos gerados para as etapas seguintes

Salvamos metadados e estatísticas produzidos durante a ingestão e a EDA para facilitar a reutilização pelo restante do grupo. Os objetos `train_ds`, `val_ds` e `test_ds` ficam prontos para consumo pelo experimento baseline.


In [ ]:
distribution.to_csv(DATA_DIR / "class_distribution.csv", index=False)
summary.to_csv(DATA_DIR / "split_summary.csv", index=False)
stats.to_csv(DATA_DIR / "image_stats.csv", index=False)
print("Artefatos salvos em data/processed/.")


## 13. Primeiro experimento — modelo baseline


## 14. Resultados preliminares

## 15. Próximos passos